In [7]:
import mediapipe as mp 
import cv2 as cv 
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import pyarrow as pa
import pyarrow.parquet as p_parquet
import os
from pathlib import Path

In [8]:
model_path = "../hand_landmarker.task"
base_options = python.BaseOptions(
    model_asset_path = model_path
)

options = vision.HandLandmarkerOptions(
    base_options = base_options,
    num_hands=2,
    min_hand_detection_confidence=0.3,
    min_tracking_confidence=0.3,
)

In [9]:
#HYPERPARAMETERS
N = 50000

In [10]:
hand_landmarks = [
    pa.field(f"{ax}{j}", pa.float32()) for j in range(1, 22) for ax in ("x", "y")
]
schema = pa.schema([
    pa.field("path", pa.string()),
    pa.field("hand_label", pa.string()),
    pa.field("frame", pa.int32()),
    *hand_landmarks
])


In [11]:
def flush(buffer : list[dict], out_path: Path, prev_num : int, next_num: int):
    if not buffer:
        return
    
    columns = {field.name : [] for field in schema}
    for row in buffer:
        for name in columns:
            columns[name].append(row.get(name))

    table = pa.table(
        {name: pa.array(vals, type=schema.field(name).type) for name, vals in columns.items()}
        )
    
    p_parquet.write_to_dataset(
        table=table,
        root_path=out_path,
        existing_data_behavior="overwrite_or_ignore",
        basename_template= f"elarna_hs_{prev_num}to{next_num}_{{i}}.parquet"

    )
    buffer.clear()


In [12]:
out_path = "../test/parquets"
buffer = []
first = 0
video_path = "/Volumes/T7/elarna/elarna_video_sorted_2020_2021"

generating_10_videos = 10
i = 0
with vision.HandLandmarker.create_from_options(options) as landmarker:
    for root, dirs, files in os.walk(video_path):
        i+=1
        if i == generating_10_videos:
            break
        for file in files:
            if file.endswith(".mp4"):
                file_path = os.path.join(root, file)
                cap = cv.VideoCapture(file_path)
                if not cap.isOpened():
                    print("Video could not be opened!")

                frame_idx = 0
                while cap.isOpened():
                    ret, frame = cap.read()

                    if not ret:
                        print("Could not read the frame. Exiting...")
                        break

                    rgb = cv.cvtColor(frame, cv.COLOR_BGR2RGB)
                    mp_image = mp.Image(mp.ImageFormat.SRGB, rgb )

                    res = landmarker.detect(mp_image)

                    if res.hand_landmarks:
                        for hand_idx, landmarks in enumerate(res.hand_landmarks):
                            wrist = landmarks[0]
                            hand_label = res.handedness[hand_idx][0].display_name
                            row = {
                                "path" : file_path,
                                "hand_label" : hand_label,
                                "frame" : frame_idx
                            }
                            for j, lm in enumerate(landmarks):
                                row[f"x{j}"] = lm.x - wrist.x
                                row[f"y{j}"] = lm.y - wrist.y
                
                            buffer.append(row)

                            if len(buffer) >= N:
                                flush(buffer, out_path, first, first+N)
                                first+= (N+1)
                    else:
                        print(f"Could not detect any landmarks on frame: {frame_idx} of {os.path.basename(file_path)}")
                    frame_idx+=1
                cap.release()
                cv.destroyAllWindows()
        flush(buffer, out_path, first, first+ len(buffer) )

I0000 00:00:1783169501.984076 2385573 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 89.4), renderer: Apple M4
W0000 00:00:1783169501.987916 2385577 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1783169501.992016 2385577 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


Could not detect any landmarks on frame: 0 of resized_cropped_23_09_1_01_02.mp4
Could not detect any landmarks on frame: 1 of resized_cropped_23_09_1_01_02.mp4
Could not detect any landmarks on frame: 2 of resized_cropped_23_09_1_01_02.mp4
Could not detect any landmarks on frame: 3 of resized_cropped_23_09_1_01_02.mp4
Could not detect any landmarks on frame: 4 of resized_cropped_23_09_1_01_02.mp4
Could not detect any landmarks on frame: 5 of resized_cropped_23_09_1_01_02.mp4
Could not detect any landmarks on frame: 6 of resized_cropped_23_09_1_01_02.mp4
Could not detect any landmarks on frame: 7 of resized_cropped_23_09_1_01_02.mp4
Could not detect any landmarks on frame: 8 of resized_cropped_23_09_1_01_02.mp4
Could not detect any landmarks on frame: 9 of resized_cropped_23_09_1_01_02.mp4
Could not detect any landmarks on frame: 10 of resized_cropped_23_09_1_01_02.mp4
Could not detect any landmarks on frame: 11 of resized_cropped_23_09_1_01_02.mp4
Could not detect any landmarks on fram

E0000 00:00:1783169622.417494 2385574 portable_clearcut_uploader.cc:90] Failed to send to clearcut: FAILED_PRECONDITION: Not valid for uploading until: 2026-07-04T18:07:42.412106+05:00
=== Source Location Trace: ===
wireless/android/play/playlog/cplusplus/portable_clearcut_uploader.cc:180


Could not detect any landmarks on frame: 10982 of resized_cropped_23_09_1_01_02.mp4
Could not detect any landmarks on frame: 10983 of resized_cropped_23_09_1_01_02.mp4
Could not detect any landmarks on frame: 11057 of resized_cropped_23_09_1_01_02.mp4
Could not detect any landmarks on frame: 11087 of resized_cropped_23_09_1_01_02.mp4
Could not detect any landmarks on frame: 11088 of resized_cropped_23_09_1_01_02.mp4
Could not detect any landmarks on frame: 11126 of resized_cropped_23_09_1_01_02.mp4
Could not detect any landmarks on frame: 11127 of resized_cropped_23_09_1_01_02.mp4
Could not detect any landmarks on frame: 11128 of resized_cropped_23_09_1_01_02.mp4
Could not detect any landmarks on frame: 11129 of resized_cropped_23_09_1_01_02.mp4
Could not detect any landmarks on frame: 11130 of resized_cropped_23_09_1_01_02.mp4
Could not detect any landmarks on frame: 11131 of resized_cropped_23_09_1_01_02.mp4
Could not detect any landmarks on frame: 11132 of resized_cropped_23_09_1_01

E0000 00:00:1783169682.420284 2385574 portable_clearcut_uploader.cc:90] Failed to send to clearcut: FAILED_PRECONDITION: Not valid for uploading until: 2026-07-04T18:07:42.412106+05:00
=== Source Location Trace: ===
wireless/android/play/playlog/cplusplus/portable_clearcut_uploader.cc:180


...

Could not detect any landmarks on frame: 206 of resized_cropped_30_09_1_01_02.mp4
Could not detect any landmarks on frame: 208 of resized_cropped_30_09_1_01_02.mp4
Could not detect any landmarks on frame: 209 of resized_cropped_30_09_1_01_02.mp4
Could not detect any landmarks on frame: 216 of resized_cropped_30_09_1_01_02.mp4
Could not detect any landmarks on frame: 217 of resized_cropped_30_09_1_01_02.mp4
Could not detect any landmarks on frame: 218 of resized_cropped_30_09_1_01_02.mp4
Could not detect any landmarks on frame: 278 of resized_cropped_30_09_1_01_02.mp4
Could not detect any landmarks on frame: 294 of resized_cropped_30_09_1_01_02.mp4
Could not detect any landmarks on frame: 295 of resized_cropped_30_09_1_01_02.mp4
Could not detect any landmarks on frame: 303 of resized_cropped_30_09_1_01_02.mp4
Could not detect any landmarks on frame: 338 of resized_cropped_30_09_1_01_02.mp4
Could not detect any landmarks on frame: 344 of resized_cropped_30_09_1_01_02.mp4
Could not detect

E0000 00:00:1783170282.488944 2385574 portable_clearcut_uploader.cc:90] Failed to send to clearcut: FAILED_PRECONDITION: Not valid for uploading until: 2026-07-04T18:07:42.412106+05:00
=== Source Location Trace: ===
wireless/android/play/playlog/cplusplus/portable_clearcut_uploader.cc:180


Could not detect any landmarks on frame: 5513 of resized_cropped_30_09_1_01_02.mp4
Could not detect any landmarks on frame: 5514 of resized_cropped_30_09_1_01_02.mp4
Could not detect any landmarks on frame: 5515 of resized_cropped_30_09_1_01_02.mp4
Could not detect any landmarks on frame: 5516 of resized_cropped_30_09_1_01_02.mp4
Could not detect any landmarks on frame: 5523 of resized_cropped_30_09_1_01_02.mp4
Could not detect any landmarks on frame: 5531 of resized_cropped_30_09_1_01_02.mp4
Could not detect any landmarks on frame: 5532 of resized_cropped_30_09_1_01_02.mp4
Could not detect any landmarks on frame: 5533 of resized_cropped_30_09_1_01_02.mp4
Could not detect any landmarks on frame: 5541 of resized_cropped_30_09_1_01_02.mp4
Could not detect any landmarks on frame: 5673 of resized_cropped_30_09_1_01_02.mp4
Could not detect any landmarks on frame: 5674 of resized_cropped_30_09_1_01_02.mp4
Could not detect any landmarks on frame: 5675 of resized_cropped_30_09_1_01_02.mp4
Coul

E0000 00:00:1783170299.326158 2385574 portable_clearcut_uploader.cc:90] Failed to send to clearcut: FAILED_PRECONDITION: Not valid for uploading until: 2026-07-04T18:07:42.412106+05:00
=== Source Location Trace: ===
wireless/android/play/playlog/cplusplus/portable_clearcut_uploader.cc:180


KeyboardInterrupt: 